# External Validation on CARD Dataset

This notebook loads the saved C4 training artifacts (feature schema, scaler, models), prepares the CARD dataset to match the training feature space, applies the trained models, and reports performance and outputs predictions.

- Leakage-free: uses the saved scaler (fitted on C4 train) and does not refit on CARD
- Strict feature alignment: uses the exact feature names and order from training
- Robust preprocessing: replicates questionnaire scoring and key feature engineering used in C4



In [ ]:
# Imports and config
import os
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, roc_curve

# Paths
ARTIFACT_DIR = '/Users/eb2007/playground/bullpy/c4_play2/models/cross_validation'
FEATURE_INFO_PATH = os.path.join(ARTIFACT_DIR, 'feature_info_original.json')
SCALER_PATH = os.path.join(ARTIFACT_DIR, 'scaler_original.joblib')
MODELS = {
    'Logistic Regression': os.path.join(ARTIFACT_DIR, 'logistic_regression_original.joblib'),
    'Random Forest': os.path.join(ARTIFACT_DIR, 'random_forest_original.joblib'),
    'XGBoost': os.path.join(ARTIFACT_DIR, 'xgboost_original.joblib'),
    'LightGBM': os.path.join(ARTIFACT_DIR, 'lightgbm_original.joblib'),
    'Gradient Boosting': os.path.join(ARTIFACT_DIR, 'gradient_boosting_original.joblib'),
}

# CARD dataset path
DATA_PATH = '/Users/eb2007/Library/CloudStorage/OneDrive-UniversityofCambridge/Documents/PhD/data/CARD_Nov2025.xlsx'

np.random.seed(42)



In [ ]:
# Load training artifacts (feature schema, scaler, models)
with open(FEATURE_INFO_PATH, 'r') as f:
    feature_info = json.load(f)

feature_names = feature_info['feature_names']
excluded_features = set(feature_info.get('excluded_features', []))
print(f"Loaded feature schema with {len(feature_names)} features")

scaler = joblib.load(SCALER_PATH)
print("Loaded saved StandardScaler (trained on C4)")

loaded_models = {}
for name, path in MODELS.items():
    if os.path.exists(path):
        loaded_models[name] = joblib.load(path)
print(f"Loaded {len(loaded_models)} trained models: {list(loaded_models.keys())}")



In [ ]:
# Load CARD dataset
print(f"Loading CARD data from: {DATA_PATH}")

# Password for encrypted Excel file
EXCEL_PASSWORD = '£ddie4ever!'

if DATA_PATH.lower().endswith(('.xlsx', '.xls')):
    try:
        # Try reading without password first
        df_card = pd.read_excel(DATA_PATH, engine='openpyxl')
        print("✅ Successfully loaded file (no password required)")
    except Exception as e:
        if 'BadZipFile' in str(type(e).__name__) or 'encrypted' in str(e).lower():
            print("⚠️  File appears to be password-protected.")
            print("Attempting to decrypt with provided password...")
            
            try:
                import msoffcrypto
                import io
                
                # Decrypt with password
                decrypted = io.BytesIO()
                with open(DATA_PATH, 'rb') as f:
                    office_file = msoffcrypto.OfficeFile(f)
                    office_file.load_key(password=EXCEL_PASSWORD)
                    office_file.decrypt(decrypted)
                    decrypted.seek(0)
                    df_card = pd.read_excel(decrypted, engine='openpyxl')
                    print("✅ Successfully decrypted and loaded file")
            except ImportError:
                print("msoffcrypto-tool not installed. Installing...")
                import subprocess
                import sys
                subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'msoffcrypto-tool'])
                print("Please re-run this cell after installation.")
                raise
            except Exception as decrypt_error:
                print(f"❌ Decryption failed: {decrypt_error}")
                print("Please check if the password is correct.")
                raise
        else:
            raise
else:
    df_card = pd.read_csv(DATA_PATH)

print(f"\nCARD dataset shape (before aggregation): {df_card.shape}")
print(f"Total columns: {len(df_card.columns)}")
print(f"First 10 columns: {list(df_card.columns[:10])}")



In [ ]:
# AGGREGATE MULTIPLE QUESTIONNAIRE ENTRIES PER PARTICIPANT
# CARD has one row per questionnaire entry, but we need one row per participant
print("="*80)
print("AGGREGATING MULTIPLE ENTRIES PER PARTICIPANT")
print("="*80)

# Identify participant ID column
# Common column names: userid, participant_id, id, subject_id, etc.
participant_id_candidates = ['userid', 'participant_id', 'id', 'subject_id', 'participant', 'user_id', 'PID']
participant_id_col = None

for col in participant_id_candidates:
    if col in df_card.columns:
        participant_id_col = col
        break

# If not found, try to find any column that looks like an ID
if participant_id_col is None:
    for col in df_card.columns:
        if any(keyword in col.lower() for keyword in ['id', 'participant', 'subject', 'user']):
            # Check if it looks like an ID (mostly unique values, numeric or string)
            unique_ratio = df_card[col].nunique() / len(df_card)
            if unique_ratio < 0.5:  # Less than 50% unique suggests it's an ID column
                participant_id_col = col
                print(f"Auto-detected participant ID column: {col}")
                break

if participant_id_col is None:
    print("ERROR: Could not identify participant ID column!")
    print("Available columns:", list(df_card.columns))
    print("\nPlease manually specify the participant ID column name in the code below:")
    print("participant_id_col = 'YOUR_COLUMN_NAME_HERE'")
    raise ValueError("Participant ID column not found. Please specify manually.")
else:
    print(f"Using participant ID column: '{participant_id_col}'")

# Check how many entries per participant
entries_per_participant = df_card.groupby(participant_id_col).size()
print(f"\nEntries per participant statistics:")
print(f"  Total participants: {df_card[participant_id_col].nunique()}")
print(f"  Total entries: {len(df_card)}")
print(f"  Mean entries per participant: {entries_per_participant.mean():.2f}")
print(f"  Median entries per participant: {entries_per_participant.median():.2f}")
print(f"  Max entries per participant: {entries_per_participant.max()}")
print(f"\n  Distribution of entries per participant:")
print(entries_per_participant.value_counts().sort_index().head(10))

# Identify columns that should be aggregated vs. kept constant per participant
# Demographics (age, sex, etc.) should be constant - take first value
# Questionnaire responses might vary - decide on strategy

# Strategy: For now, we'll take the MOST RECENT entry per participant
# If there's a date/timestamp column, use it; otherwise use the last row

# Look for date/timestamp columns
date_col_candidates = [c for c in df_card.columns if any(kw in c.lower() for kw in ['date', 'time', 'timestamp', 'created', 'submitted'])]
if date_col_candidates:
    date_col = date_col_candidates[0]
    print(f"\nFound date/timestamp column: '{date_col}' - will use most recent entry per participant")
    # Convert to datetime if possible
    try:
        df_card[date_col] = pd.to_datetime(df_card[date_col], errors='coerce')
        df_card = df_card.sort_values(by=[participant_id_col, date_col])
    except:
        print(f"  Warning: Could not parse {date_col} as datetime, using last row instead")
        date_col = None
else:
    print("\nNo date/timestamp column found - will use last row per participant")
    date_col = None

# Aggregate to one row per participant
if date_col:
    # Take most recent entry per participant
    df_card = df_card.groupby(participant_id_col).last().reset_index()
else:
    # Take last row per participant (assuming data is already sorted)
    df_card = df_card.groupby(participant_id_col).last().reset_index()

print(f"\nAfter aggregation: {df_card.shape}")
print(f"  Participants: {df_card[participant_id_col].nunique()}")
print(f"  Rows: {len(df_card)}")
print(f"  ✅ Successfully aggregated to one row per participant")

# Rename participant ID column to 'userid' for consistency with C4
if participant_id_col != 'userid':
    df_card = df_card.rename(columns={participant_id_col: 'userid'})
    print(f"  Renamed '{participant_id_col}' to 'userid'")

print("\n" + "="*80)



In [ ]:
# COMPREHENSIVE DATA EXPLORATION: Summarize CARD dataset structure
print("="*80)
print("DATA EXPLORATION: Understanding CARD Dataset Structure")
print("="*80)

# First, we need to reload the original data to see all entries (before aggregation)
# This will help us understand what tests/questionnaires are available
print("\n1. LOADING ORIGINAL DATA (before aggregation) to explore test structure...")
if DATA_PATH.lower().endswith(('.xlsx', '.xls')):
    try:
        df_card_original = pd.read_excel(DATA_PATH, engine='openpyxl')
    except:
        import msoffcrypto
        import io
        decrypted = io.BytesIO()
        with open(DATA_PATH, 'rb') as f:
            office_file = msoffcrypto.OfficeFile(f)
            office_file.load_key(password=EXCEL_PASSWORD)
            office_file.decrypt(decrypted)
            decrypted.seek(0)
            df_card_original = pd.read_excel(decrypted, engine='openpyxl')
else:
    df_card_original = pd.read_csv(DATA_PATH)

print(f"   Original dataset: {df_card_original.shape[0]} rows, {df_card_original.shape[1]} columns")

# Identify participant ID
participant_id_col = 'VolunteerID' if 'VolunteerID' in df_card_original.columns else 'userid'

print("\n2. QUESTIONNAIRE/TEST SUMMARY:")
if 'TestName' in df_card_original.columns:
    print(f"\n   Available Tests/Questionnaires:")
    test_counts = df_card_original['TestName'].value_counts()
    print(f"   Total unique tests: {df_card_original['TestName'].nunique()}")
    print(f"\n   Test names and frequencies:")
    for test_name, count in test_counts.items():
        pct = (count / len(df_card_original) * 100)
        print(f"     {test_name}: {count:,} entries ({pct:.1f}%)")
    
    print(f"\n   Participants with each test:")
    participants_per_test = df_card_original.groupby('TestName')[participant_id_col].nunique().sort_values(ascending=False)
    for test_name, n_participants in participants_per_test.items():
        pct = (n_participants / df_card_original[participant_id_col].nunique() * 100)
        print(f"     {test_name}: {n_participants:,} participants ({pct:.1f}% of all participants)")
    
    # Check for expected questionnaire names
    test_names_lower = [str(t).lower() for t in df_card_original['TestName'].unique()]
    print(f"\n   Looking for expected questionnaires (SPQ, EQ, SQR, AQ):")
    expected_tests = ['spq', 'eq', 'sqr', 'aq']
    for exp_test in expected_tests:
        matching = [t for t in test_names_lower if exp_test in t]
        if matching:
            print(f"     ✓ Found {exp_test.upper()}-related tests: {matching}")
        else:
            print(f"     ✗ No {exp_test.upper()}-related tests found")
    
    # Score distributions by test
    if 'Score' in df_card_original.columns:
        print(f"\n   Score distributions by test:")
        for test_name in test_counts.index[:10]:  # Top 10 tests
            test_scores = df_card_original[df_card_original['TestName'] == test_name]['Score']
            if test_scores.notna().sum() > 0:
                print(f"\n     {test_name}:")
                print(f"       Mean: {test_scores.mean():.2f}")
                print(f"       Median: {test_scores.median():.2f}")
                print(f"       Min: {test_scores.min():.2f}")
                print(f"       Max: {test_scores.max():.2f}")
                print(f"       Std: {test_scores.std():.2f}")
                print(f"       Missing: {test_scores.isna().sum()} ({test_scores.isna().sum()/len(test_scores)*100:.1f}%)")
else:
    print("   ⚠️  No 'TestName' column found - cannot identify questionnaires")

print("\n3. PARTICIPANT COVERAGE:")
participant_test_counts = df_card_original.groupby(participant_id_col)['TestName'].nunique() if 'TestName' in df_card_original.columns else pd.Series()
if len(participant_test_counts) > 0:
    print(f"   Tests per participant:")
    print(f"     Mean: {participant_test_counts.mean():.2f}")
    print(f"     Median: {participant_test_counts.median():.2f}")
    print(f"     Min: {participant_test_counts.min()}")
    print(f"     Max: {participant_test_counts.max()}")
    print(f"\n   Distribution:")
    print(participant_test_counts.value_counts().sort_index().head(10))

print("\n4. DEMOGRAPHICS SUMMARY:")
demographic_cols = ['AgeWhenTestCompleted', 'Sex', 'Ethnicity', 'Occupation', 'Country', 'Handedness']
for col in demographic_cols:
    if col in df_card_original.columns:
        print(f"\n   {col}:")
        if df_card_original[col].dtype == 'object':
            print(f"     Unique values: {df_card_original[col].nunique()}")
            print(f"     Top values:")
            print(df_card_original[col].value_counts().head(5))
        else:
            print(f"     Mean: {df_card_original[col].mean():.2f}")
            print(f"     Median: {df_card_original[col].median():.2f}")
            print(f"     Range: {df_card_original[col].min():.2f} - {df_card_original[col].max():.2f}")
        missing = df_card_original[col].isna().sum()
        print(f"     Missing: {missing} ({missing/len(df_card_original)*100:.1f}%)")

print("\n5. DIAGNOSIS INFORMATION:")
diagnosis_cols = [c for c in df_card_original.columns if any(kw in c.lower() for kw in ['diagnosis', 'autism', 'asc'])]
if diagnosis_cols:
    print(f"   Diagnosis-related columns: {diagnosis_cols}")
    for col in diagnosis_cols:
        print(f"\n   {col}:")
        print(df_card_original[col].value_counts().head(10))
        missing = df_card_original[col].isna().sum()
        print(f"     Missing: {missing} ({missing/len(df_card_original)*100:.1f}%)")

print("\n" + "="*80)
print("EXPLORATION COMPLETE")
print("="*80)



In [ ]:
# INSPECT CARD DATA STRUCTURE
# Compare with expected C4 format to identify preprocessing needs
print("="*80)
print("DATA INSPECTION: Comparing CARD structure with expected C4 format")
print("="*80)

print("\n1. COLUMN NAMES:")
print(f"   CARD has {len(df_card.columns)} columns")
print(f"   First 20 columns: {list(df_card.columns[:20])}")

# Check for expected questionnaire columns
expected_spq = [f'spq_{i}' for i in range(1, 11)]
expected_eq = [f'eq_{i}' for i in range(1, 11)]



  
expected_sqr = [f'sqr_{i}' for i in range(1, 11)]
expected_aq = [f'aq_{i}' for i in range(1, 11)]

found_spq = [c for c in df_card.columns if any(c.lower().startswith(f'spq_{i}') or c.lower() == f'spq_{i}' for i in range(1, 11))]
found_eq = [c for c in df_card.columns if any(c.lower().startswith(f'eq_{i}') or c.lower() == f'eq_{i}' for i in range(1, 11))]
found_sqr = [c for c in df_card.columns if any(c.lower().startswith(f'sqr_{i}') or c.lower() == f'sqr_{i}' for i in range(1, 11))]
found_aq = [c for c in df_card.columns if any(c.lower().startswith(f'aq_{i}') or c.lower() == f'aq_{i}' for i in range(1, 11))]

print(f"\n   SPQ columns found: {len(found_spq)} - {found_spq[:5]}...")
print(f"   EQ columns found: {len(found_eq)} - {found_eq[:5]}...")
print(f"   SQR columns found: {len(found_sqr)} - {found_sqr[:5]}...")
print(f"   AQ columns found: {len(found_aq)} - {found_aq[:5]}...")

# Check for demographic columns
demographic_keywords = ['age', 'sex', 'gender', 'occupation', 'education', 'handedness', 'country']
found_demographics = {kw: [c for c in df_card.columns if kw.lower() in c.lower()] for kw in demographic_keywords}
print(f"\n   Demographics found:")
for kw, cols in found_demographics.items():
    if cols:
        print(f"     {kw}: {cols[:3]}...")

# Check for diagnosis/autism columns
diagnosis_keywords = ['diagnosis', 'autism', 'asd']
found_diagnosis = {kw: [c for c in df_card.columns if kw.lower() in c.lower()] for kw in diagnosis_keywords}
print(f"\n   Diagnosis columns found:")
for kw, cols in found_diagnosis.items():
    if cols:
        print(f"     {kw}: {cols[:5]}...")

print("\n2. DATA TYPES:")
print(df_card.dtypes.value_counts())

print("\n3. SAMPLE VALUES:")
# Show sample values for questionnaire columns if found
if found_spq:
    print(f"\n   Sample SPQ values (first column):")
    print(f"     {df_card[found_spq[0]].value_counts().head()}")
if found_eq:
    print(f"\n   Sample EQ values (first column):")
    print(f"     {df_card[found_eq[0]].value_counts().head()}")

# CRITICAL: Check TestName column to understand data structure
if 'TestName' in df_card.columns:
    print(f"\n   TestName column values (this is KEY for understanding CARD structure):")
    print(f"     Unique test names: {df_card['TestName'].nunique()}")
    print(f"     Test names: {df_card['TestName'].value_counts().head(20)}")
    print(f"\n   Sample rows with TestName and Score:")
    print(df_card[['userid', 'TestName', 'Score']].head(20))
    
if 'Score' in df_card.columns:
    print(f"\n   Score column statistics:")
    print(f"     {df_card['Score'].describe()}")
    print(f"     Sample Score values: {df_card['Score'].value_counts().head(10)}")

print("\n4. MISSING VALUES:")
missing_counts = df_card.isnull().sum()
missing_pct = (missing_counts / len(df_card) * 100).round(2)
missing_summary = pd.DataFrame({'Missing Count': missing_counts, 'Missing %': missing_pct})
missing_summary = missing_summary[missing_summary['Missing Count'] > 0].sort_values('Missing Count', ascending=False)
if len(missing_summary) > 0:
    print(f"   Columns with missing values: {len(missing_summary)}")
    print(missing_summary.head(10))
else:
    print("   No missing values found")

print("\n" + "="*80)
print("INSPECTION COMPLETE - Review output above to identify preprocessing needs")
print("="*80)



In [ ]:
# PREPROCESSING: Transform CARD data to match C4 format
# This cell handles common differences: column name mapping, value recoding, etc.
print("="*80)
print("PREPROCESSING: Aligning CARD data structure with C4 format")
print("="*80)

# Create a mapping dictionary for column name standardization
# Add mappings here based on inspection results above
column_mapping = {}

# Common variations for questionnaire columns
# Example: if CARD uses "SPQ1" instead of "spq_1", add: column_mapping['SPQ1'] = 'spq_1'
# You may need to adjust these based on actual CARD column names

# Apply column name mappings
if column_mapping:
    df_card = df_card.rename(columns=column_mapping)
    print(f"Renamed {len(column_mapping)} columns")

# Standardize questionnaire column names (handle case variations and spacing)
# Convert any variation like "SPQ_1", "spq1", "SPQ 1" to "spq_1"
for i in range(1, 11):
    # Find variations of spq columns
    spq_variations = [c for c in df_card.columns if any([
        c.lower().replace(' ', '_').replace('-', '_') == f'spq_{i}',
        c.lower().replace(' ', '').replace('-', '') == f'spq{i}',
        c.lower().replace(' ', '_').replace('-', '_').endswith(f'spq_{i}'),
    ])]
    if spq_variations and f'spq_{i}' not in df_card.columns:
        df_card[f'spq_{i}'] = df_card[spq_variations[0]]
        print(f"  Mapped {spq_variations[0]} -> spq_{i}")
    
    # Find variations of eq columns
    eq_variations = [c for c in df_card.columns if any([
        c.lower().replace(' ', '_').replace('-', '_') == f'eq_{i}',
        c.lower().replace(' ', '').replace('-', '') == f'eq{i}',
        c.lower().replace(' ', '_').replace('-', '_').endswith(f'eq_{i}'),
    ])]
    if eq_variations and f'eq_{i}' not in df_card.columns:
        df_card[f'eq_{i}'] = df_card[eq_variations[0]]
        print(f"  Mapped {eq_variations[0]} -> eq_{i}")
    
    # Find variations of sqr columns
    sqr_variations = [c for c in df_card.columns if any([
        c.lower().replace(' ', '_').replace('-', '_') == f'sqr_{i}',
        c.lower().replace(' ', '').replace('-', '') == f'sqr{i}',
        c.lower().replace(' ', '_').replace('-', '_').endswith(f'sqr_{i}'),
    ])]
    if sqr_variations and f'sqr_{i}' not in df_card.columns:
        df_card[f'sqr_{i}'] = df_card[sqr_variations[0]]
        print(f"  Mapped {sqr_variations[0]} -> sqr_{i}")
    
    # Find variations of aq columns
    aq_variations = [c for c in df_card.columns if any([
        c.lower().replace(' ', '_').replace('-', '_') == f'aq_{i}',
        c.lower().replace(' ', '').replace('-', '') == f'aq{i}',
        c.lower().replace(' ', '_').replace('-', '_').endswith(f'aq_{i}'),
    ])]
    if aq_variations and f'aq_{i}' not in df_card.columns:
        df_card[f'aq_{i}'] = df_card[aq_variations[0]]
        print(f"  Mapped {aq_variations[0]} -> aq_{i}")

# Handle demographic column name variations
demographic_mappings = {
    'gender': 'sex',
    'Gender': 'sex',
    'GENDER': 'sex',
    'Age': 'age',
    'AGE': 'age',
    'Occupation': 'occupation',
    'OCCUPATION': 'occupation',
    'Education': 'education',
    'EDUCATION': 'education',
}

for old_name, new_name in demographic_mappings.items():
    if old_name in df_card.columns and new_name not in df_card.columns:
        df_card[new_name] = df_card[old_name]
        print(f"  Mapped {old_name} -> {new_name}")

# Ensure occupation is string type for later processing
if 'occupation' in df_card.columns:
    df_card['occupation'] = df_card['occupation'].astype(str)

# Handle value recoding if needed
# C4 uses: questionnaire items are 1-4, need to check if CARD uses different encoding
# Example: if CARD uses 0-3 instead of 1-4, add recoding here

print(f"\nAfter preprocessing: {df_card.shape}")
print(f"Columns now include: {[c for c in df_card.columns if any(x in c.lower() for x in ['spq', 'eq', 'sqr', 'aq', 'age', 'sex'])]}")

print("\n" + "="*80)
print("PREPROCESSING COMPLETE")
print("="*80)



In [ ]:
# Questionnaire scoring (replicate C4 rules)
# SPQ-10 items: 1->3, 2->2, 3->1, 4->0; total 0-30
spq_cols = [c for c in df_card.columns if c.lower().startswith('spq_')]
for c in spq_cols:
    df_card[c] = df_card[c].map({1: 3, 2: 2, 3: 1, 4: 0})
if spq_cols:
    df_card['spq_total'] = df_card[spq_cols].sum(axis=1)

# EQ-10 items: 1->1, 2/3/4->0; total 0-10
eq_cols = [c for c in df_card.columns if c.lower().startswith('eq_')]
for c in eq_cols:
    df_card[c] = df_card[c].map({1: 1, 2: 0, 3: 0, 4: 0})
if eq_cols:
    df_card['eq_total'] = df_card[eq_cols].sum(axis=1)

# SQR-10 items: 1->1, 2/3/4->0; total 0-10
sqr_cols = [c for c in df_card.columns if c.lower().startswith('sqr_')]
for c in sqr_cols:
    df_card[c] = df_card[c].map({1: 1, 2: 0, 3: 0, 4: 0})
if sqr_cols:
    df_card['sqr_total'] = df_card[sqr_cols].sum(axis=1)

# AQ-10 items: 1->1, 2/3/4->0; total 0-10
aq_cols = [c for c in df_card.columns if c.lower().startswith('aq_')]
for c in aq_cols:
    df_card[c] = df_card[c].map({1: 1, 2: 0, 3: 0, 4: 0})
if aq_cols:
    df_card['aq_total'] = df_card[aq_cols].sum(axis=1)

print('Scoring complete:')
print({
    'spq_items': len(spq_cols),
    'eq_items': len(eq_cols),
    'sqr_items': len(sqr_cols),
    'aq_items': len(aq_cols)
})



In [ ]:
# Feature engineering to match training
# Age groups and transforms
if 'age' in df_card.columns:
    df_card['sqrt_age'] = np.sqrt(np.clip(df_card['age'], a_min=0, a_max=None))
    df_card['age_group_19-30'] = ((df_card['age'] >= 19) & (df_card['age'] <= 30)).astype(int)
    df_card['age_group_31-45'] = ((df_card['age'] >= 31) & (df_card['age'] <= 45)).astype(int)
    df_card['age_group_46-60'] = ((df_card['age'] >= 46) & (df_card['age'] <= 60)).astype(int)
    df_card['age_group_61+'] = (df_card['age'] >= 61).astype(int)
else:
    df_card['sqrt_age'] = 0.0
    df_card['age_group_19-30'] = 0
    df_card['age_group_31-45'] = 0
    df_card['age_group_46-60'] = 0
    df_card['age_group_61+'] = 0

# sex_num mapping as used in C4 (fallback to 0 if missing)
if 'sex' in df_card.columns:
    df_card['sex_num'] = df_card['sex'].map({1: 0, 2: 1, 3: 2, 4: 3}).fillna(0).astype(int)
else:
    df_card['sex_num'] = 0

# STEM occupation
if 'occupation' in df_card.columns:
    df_card['is_stem_occupation'] = df_card['occupation'].str.contains(
        'science|technology|engineering|math|computer|software|data|research', case=False, na=False
    ).astype(int)
else:
    df_card['is_stem_occupation'] = 0

# Interactions and ratios consistent with feature_info
if {'age', 'eq_total'}.issubset(df_card.columns):
    df_card['age_x_eq'] = df_card['age'] * df_card['eq_total']
else:
    df_card['age_x_eq'] = 0.0

if {'eq_total', 'sqr_total'}.issubset(df_card.columns):
    df_card['eq_sqr_ratio'] = df_card['eq_total'] / (df_card['sqr_total'].replace(0, np.nan))
    df_card['eq_sqr_ratio'] = df_card['eq_sqr_ratio'].replace([np.inf, -np.inf], np.nan).fillna(0.0)
else:
    df_card['eq_sqr_ratio'] = 0.0

# d_score (difference between SQR and EQ; sign consistent with prior code)
if {'sqr_total', 'eq_total'}.issubset(df_card.columns):
    df_card['d_score'] = df_card['sqr_total'] - df_card['eq_total']
else:
    df_card['d_score'] = 0.0



In [ ]:
# Build aligned feature matrix in exact training order
X_card = pd.DataFrame(index=df_card.index)
missing_from_card = []
for fname in feature_names:
    if fname in df_card.columns:
        X_card[fname] = df_card[fname]
    else:
        # Create missing feature as 0
        X_card[fname] = 0
        missing_from_card.append(fname)

# Basic type coercion and missing handling
for c in X_card.columns:
    if X_card[c].dtype == 'object':
        X_card[c] = pd.Categorical(X_card[c]).codes

X_card = X_card.apply(pd.to_numeric, errors='coerce')
num_missing = int(X_card.isnull().sum().sum())
if num_missing > 0:
    X_card = X_card.fillna(X_card.median(numeric_only=True))

print(f"Aligned feature matrix shape: {X_card.shape}")
if missing_from_card:
    print(f"Note: Missing {len(missing_from_card)} features in CARD, filled with 0: {missing_from_card[:10]}...")



In [ ]:
# Apply saved scaler (no refit)
X_card_scaled = scaler.transform(X_card.values)

# Predict with each model
results = {}
probas = {}
for name, model in loaded_models.items():
    y_proba = model.predict_proba(X_card_scaled)[:, 1]
    y_pred = (y_proba >= 0.5).astype(int)
    probas[name] = y_proba
    results[name] = {'n': len(y_pred)}

print('Predictions generated for models:', list(results.keys()))



In [ ]:
# Evaluate if ground-truth label present
metrics_df = None
label_col_candidates = ['autism_target', 'diagnosis_autism', 'has_autism']
label_col = next((c for c in label_col_candidates if c in df_card.columns), None)

if label_col is not None:
    y_true = df_card[label_col].astype(int).clip(0, 1).values
    for name, model in loaded_models.items():
        y_proba = probas[name]
        y_pred = (y_proba >= 0.5).astype(int)
        results[name].update({
            'accuracy': accuracy_score(y_true, y_pred),
            'precision': precision_score(y_true, y_pred, zero_division=0),
            'recall': recall_score(y_true, y_pred, zero_division=0),
            'f1': f1_score(y_true, y_pred, zero_division=0),
            'auc': roc_auc_score(y_true, y_proba)
        })
    metrics_df = pd.DataFrame(results).T
    print('External validation metrics (CARD):')
    print(metrics_df.round(4).sort_values('auc', ascending=False))
else:
    print('No label column found; skipping metrics. Saving predictions only.')



In [ ]:
# Save outputs
os.makedirs('/Users/eb2007/playground/bullpy/c4_play2/data/processed', exist_ok=True)

pred_df = pd.DataFrame({'userid': df_card['userid'] if 'userid' in df_card.columns else np.arange(len(df_card))})
for name, y_proba in probas.items():
    pred_df[f'proba_{name.replace(" ", "_").lower()}'] = y_proba

pred_path = '/Users/eb2007/playground/bullpy/c4_play2/data/processed/card_external_predictions.csv'
pred_df.to_csv(pred_path, index=False)
print(f"Predictions saved to: {pred_path}")

feat_used_path = '/Users/eb2007/playground/bullpy/c4_play2/data/processed/card_features_used.json'
with open(feat_used_path, 'w') as f:
    json.dump({'feature_names': feature_names, 'missing_filled_zero': missing_from_card}, f, indent=2)
print(f"Feature alignment info saved to: {feat_used_path}")

if metrics_df is not None:
    metrics_path = '/Users/eb2007/playground/bullpy/c4_play2/data/processed/card_external_metrics.csv'
    metrics_df.to_csv(metrics_path)
    print(f"Metrics saved to: {metrics_path}")

